# Telco customer churn: data understanding

## Project goals

1. Describe the customer base and the overall churn rate.
2. Analyse how contract type, tenure, services and payment method relate to churn.
3. Build an interpretable logistic-regression baseline that flags customers at risk.

This notebook covers data structure and data quality only. No model is trained here.

**Source:** the public Kaggle dataset `blastchar/telco-customer-churn`.  
**Grain:** one row per customer.  
**Target:** `Churn`, where `Yes` means the customer left and `No` means they stayed.

## 1. Loading the data

The notebook lives in `notebooks/`, so the raw CSV is read from `../data/raw/`.

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# First rows, to confirm the file loaded and to see what the fields look like
display(df.head())

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Structure

Size, column names, column types and the missing values pandas can detect on its own. `info()` shows the non-null count and type of every column.

In [2]:
print(f"Size: {df.shape[0]} rows, {df.shape[1]} columns")

print("Columns:")
print(df.columns.tolist())

df.info()

# Missing values per column
display(df.isna().sum().rename("missing_count").to_frame())

Size: 7043 rows, 21 columns
Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    704

,missing_count
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


## 3. Data quality

`isna()` finds no missing values, but `TotalCharges` is read as text although it should be numeric, so blank strings need checking. Duplicate customer IDs and fully duplicated rows are checked as well.

In [3]:
# Rows where TotalCharges is a single space
empty_total_charges_condition = df["TotalCharges"] == " "

empty_total_charges_count = empty_total_charges_condition.sum()
print("Rows with a blank TotalCharges:", empty_total_charges_count)

# Customer ID, tenure and charges of those rows
display(
    df.loc[
        empty_total_charges_condition,
        ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]
    ]
)

# customerID should identify one customer; also check fully identical rows
customer_id_duplicate_count = df["customerID"].duplicated().sum()
full_row_duplicate_count = df.duplicated().sum()

print("Duplicate customerID:", customer_id_duplicate_count)
print("Fully duplicated rows:", full_row_duplicate_count)

Rows with a blank TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


Duplicate customerID: 0
Fully duplicated rows: 0


## 4. The target, `Churn`

In [4]:
churn_count = df["Churn"].value_counts()
churn_percentage = df["Churn"].value_counts(normalize=True) * 100

display(churn_count.rename("customer_count").to_frame())
display(churn_percentage.round(2).rename("percentage").to_frame())

,customer_count
Churn,
No,5174
Yes,1869


,percentage
Churn,
No,73.46
Yes,26.54


## 5. Numeric fields as loaded

`tenure` and `MonthlyCharges` are already numeric.

In [5]:
display(df[["tenure", "MonthlyCharges"]].describe())

,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


## 6. Cleaning `TotalCharges`

The blank values belong to new customers with `tenure = 0`, who have not been billed yet. The column is converted to numeric first; `errors="coerce"` turns the blanks into `NaN`. After confirming the result, they are filled with 0, which is what the business meaning implies. The raw `df` is left untouched; cleaning happens on the copy `clean_df`.

This fill is for descriptive analysis only. In the modeling notebooks the blanks stay missing and are imputed with a median learned from training data.

In [6]:
clean_df = df.copy()

# Text to numeric; blanks become NaN
clean_df["TotalCharges"] = pd.to_numeric(
    clean_df["TotalCharges"],
    errors="coerce"
)

print("Type after conversion:", clean_df["TotalCharges"].dtype)
print("NaN after conversion:", clean_df["TotalCharges"].isna().sum())

# All of these rows have tenure 0, so total charges are set to 0
clean_df["TotalCharges"] = clean_df["TotalCharges"].fillna(0)

print("NaN after filling:", clean_df["TotalCharges"].isna().sum())

Type after conversion: float64
NaN after conversion: 11
NaN after filling: 0


## 7. Verifying the cleaned data

Types, non-null counts and summary statistics of the three numeric fields, to confirm that no customer was dropped and no missing value is left.

In [7]:
clean_df.info()

display(
    clean_df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()
)

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000
mean,32.371149,64.761692,2279.734304
std,24.559481,30.090047,2266.794470
min,0.000000,18.250000,0.000000
25%,9.000000,35.500000,398.550000
50%,29.000000,70.350000,1394.550000
75%,55.000000,89.850000,3786.600000
max,72.000000,118.750000,8684.800000


## 8. Summary

- The data has **7,043 customers and 21 columns**, one row per customer.
- `customerID` has no duplicates and no row is fully duplicated.
- `Churn` is the target: 1,869 customers churned, a rate of about **26.54%**.
- `TotalCharges` was read as text because it contains **11 blank values**.
- All 11 belong to customers with `tenure` 0 who have not been billed yet, so the blanks were converted and filled with 0.
- Average tenure is about **32.37 months**, average monthly charge about **64.76**, and average total charges about **2,279.73**.

## Next

Exploratory analysis: churn rates by contract type, tenure, internet service and payment method. No causal claim and no model in that step either.